# FINA4030A — Lab 2
## Measuring capability and reliability

**Class 2.** Submit this notebook by 23:59 on **24 September**.

Last week you gave a task to a machine at three levels of autonomy. This week you ask a narrower question, and the answer is the point of the whole course:

> When you send the same question twice, do you get the same answer?

You already know the answer for a spreadsheet. You are about to find out the answer for the thing that is increasingly being put in the spreadsheet's place.

**What you will do.** Send one small, completely specified valuation task ten times, with the settings that are supposed to make the output deterministic. Measure what comes back. Then compute the same valuation in Python, ten times, and compare the two distributions.

**What is marked.** Not the numbers. The verification you did, what you found, and whether your stated confidence turned out to be right. A notebook reporting a clean result that was in fact wrong scores badly. A notebook reporting a problem, correctly diagnosed, scores well.


In [ ]:
# Setup. Run this first.
# Pulls the shared course client from GitHub so everyone is on the same version.

REQUIRED_CLIENT = "1.0"          # bump when the client changes
REPO = "https://raw.githubusercontent.com/fy-ericlam/fina4030a/main"

import importlib, sys, urllib.request

urllib.request.urlretrieve(f"{REPO}/fina4030a.py", "fina4030a.py")
urllib.request.urlretrieve(f"{REPO}/labs/lab02_cached_responses.json",
                           "lab02_cached_responses.json")

# Downloading the file is not enough: if the module was already imported, Python
# keeps the old copy in memory. Drop it and reload.
sys.modules.pop("fina4030a", None)
import fina4030a
importlib.reload(fina4030a)

if fina4030a.__version__ < REQUIRED_CLIENT:
    print(f"!! Loaded client v{fina4030a.__version__}, but this lab needs "
          f"v{REQUIRED_CLIENT}.")
    print("   Runtime > Restart session, then run this cell again.")
    print("   If it still says the old version, the repo has not been updated.")
else:
    print(f"client v{fina4030a.__version__} loaded")

# --- your details -----------------------------------------------------------
NAME       = ""          # e.g. "CHAN Tai Man"
STUDENT_ID = ""          # e.g. "1155123456"

# --- provider ---------------------------------------------------------------
# Leave as is unless told otherwise in class. If live access fails during the
# lab, switch to the cached line below and carry on — but see the note at the
# end about what that does and does not let you claim.
fina4030a.configure(provider="cuhk_portal")   # CUHK APIP, model gpt-5.4-mini
# fina4030a.configure(provider="cached", cache_path="lab02_cached_responses.json")

fina4030a.verify()

> **If the check above failed, read the step it stopped on — it names the layer.**
>
> **Step 1 failed (401).** Your APIP subscription key is missing or wrong. Check Colab's secrets panel (key icon, left sidebar): the secret must be named exactly `CUHK_APIM_KEY` and **notebook access must be toggled on** for this notebook. Never paste the key into a cell — it would be saved into the notebook and into anything you submit.
>
> **Step 2 failed.** The model named in the client is not one your token can reach. The check prints the ones that are — run `fina4030a.configure(model="<one of them>")` and then `fina4030a.verify()` again.
>
> **It reports an old client version.** Downloading the file does not replace a module already in memory. **Runtime → Restart session**, then run the setup cell again.
>
> **Nothing loads at all.** Turn your VPN off or reconnect it, or switch to your phone's hotspot.
>
> **Still stuck after five minutes?** Switch the provider line to the `cached` one and carry on with the lab. You will not have measured anything yourself, and you must say so in your findings — but you will still learn what the lab is for, and we can fix your access afterwards.

---

## The task

Everything the model needs is in the prompt. There is nothing to look up, nothing to assume, and exactly one defensible answer. That is deliberate: if answers still vary, the variation cannot be blamed on ambiguity in the question.

Read the prompt before you run it. You should be able to work out roughly what the answer ought to be.

In [ ]:
TASK = """You are valuing a company. Use only the figures given.

Free cash flow, HK$ millions, end of each year:
  Year 1: 120
  Year 2: 135
  Year 3: 150
  Year 4: 162
  Year 5: 170

WACC: 9.5%
Terminal growth rate after Year 5: 2.5%
Value the terminal value using the Gordon growth model as at the end of Year 5,
then discount it back to today along with the explicit forecast cash flows.

Compute the enterprise value today, in HK$ millions.

Show your working briefly, then end your reply with a single final line in
exactly this format, with no other text on that line:
ANSWER: <number>"""

print(TASK)

---

## Ten runs, identical settings

`temperature=0.0` is what people reach for when they want a deterministic answer. Watch whether it delivers one.

**Check the setup output above before you go on.** If it says the request was adapted and that temperature was *not accepted*, then this model will not let you ask for determinism at all — the parameter is refused outright. That is a stronger version of the same finding, and you should say so explicitly in your write-up rather than pretending you ran a controlled test.

**This cell takes about two and a half minutes, and that is deliberate.** The University's Starter plan allows five calls a minute, so the client paces itself to stay inside that. Do not interrupt it and do not add your own pauses.

You have 100 calls a week on this plan, renewed weekly, with an email warning at 75%. One full run of this lab uses eleven. Budget accordingly if you want to re-run — and if you do exhaust it, switch to the cached responses and say so in your findings.

If a call fails, the client will tell you why.

In [ ]:
N_RUNS = 10
responses = []

for i in range(N_RUNS):
    try:
        r = fina4030a.complete(TASK, temperature=0.0, max_tokens=400)
    except fina4030a.ModelError as e:
        print(f"run {i+1}: FAILED — {str(e).splitlines()[0]}")
        r = ""
    responses.append(r)
    val = fina4030a.extract_answer(r)
    print(f"run {i+1:>2}/{N_RUNS}: {'no parseable ANSWER line' if val is None else f'{val:,.1f}'}")

print(f"\n{len(responses)} responses collected.")

---

## What came back

Two things to notice, and the second matters more.

First, format compliance: the `ANSWER:` line was requested in an exact format. Where it is missing or unparseable, that is an observation about the model, not a gap in your data, and it belongs in your findings.

Second, and more important: **count the distinct answers.** Not the spread, not the standard deviation — the number of different values. Then ask yourself what you expected that number to be before you ran the cell.

In [ ]:
import pandas as pd, numpy as np

vals = [fina4030a.extract_answer(r) for r in responses]
df = pd.DataFrame({
    "run": range(1, len(responses) + 1),
    "answer": vals,
    "chars": [len(r) for r in responses],
    "parsed": [v is not None for v in vals],
})
display(df)

parsed = df["answer"].dropna()
n_fail = int((~df["parsed"]).sum())

if len(parsed) == 0:
    print("Nothing parsed. That is a finding in itself — record it.")
else:
    print(f"parsed        {len(parsed)} of {len(df)}   "
          f"({n_fail} did not follow the requested format)")
    print(f"distinct      {parsed.nunique()} different answers")
    print(f"min / max     {parsed.min():,.1f}  /  {parsed.max():,.1f}")
    print(f"spread        {parsed.max() - parsed.min():,.1f}  "
          f"({(parsed.max() - parsed.min()) / parsed.median() * 100:.1f}% of median)")
    print(f"mean / median {parsed.mean():,.1f}  /  {parsed.median():,.1f}")
    print(f"std dev       {parsed.std():,.1f}")

### Look at it

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))

ax[0].scatter(df["run"], df["answer"], s=48, color="#1f3b73", zorder=3)
if len(parsed):
    ax[0].axhline(parsed.median(), color="#b03a2e", lw=1, ls="--",
                  label=f"median {parsed.median():,.0f}")
    ax[0].legend(fontsize=8)
ax[0].set_xlabel("run"); ax[0].set_ylabel("enterprise value (HK$m)")
ax[0].set_title("Same prompt, same settings, ten times", fontsize=10)
ax[0].grid(alpha=.25)

if len(parsed) > 1:
    ax[1].hist(parsed, bins=min(10, parsed.nunique()), color="#1f3b73", alpha=.8)
ax[1].set_xlabel("enterprise value (HK$m)"); ax[1].set_ylabel("count")
ax[1].set_title("Distribution of answers", fontsize=10)
ax[1].grid(alpha=.25)

fig.tight_layout(); plt.show()

---

## Now the same calculation, deterministically

Here is the valuation in Python. Read it and satisfy yourself it implements what the prompt asked for. Then run it ten times.

In [ ]:
def enterprise_value(fcf, wacc, g):
    """PV of explicit FCF plus PV of a Gordon-growth terminal value at year N."""
    pv_explicit = sum(cf / (1 + wacc) ** t for t, cf in enumerate(fcf, start=1))
    tv          = fcf[-1] * (1 + g) / (wacc - g)          # value at end of year N
    pv_tv       = tv / (1 + wacc) ** len(fcf)             # discounted N years
    return pv_explicit + pv_tv

FCF, WACC, G = [120, 135, 150, 162, 170], 0.095, 0.025

runs = [enterprise_value(FCF, WACC, G) for _ in range(10)]
truth = runs[0]

print(f"ten runs: {len(set(runs))} distinct value(s)")
print(f"value:    {truth:,.4f}")
print(f"spread:   {max(runs) - min(runs):.10f}")

### Now look again at the model's answers: noise, or two methods?

A standard deviation treats every difference as the same kind of thing. Before you accept that summary, look at whether the answers are *spread* or *grouped*.

Sort them and look at the gaps. If one gap is far larger than the rest, you are not looking at noise around a single answer — you are looking at the model taking two different routes on different runs, and averaging them would be meaningless.

In [ ]:
import numpy as np

vals = sorted(parsed.tolist())
gaps = [(vals[i+1] - vals[i], i) for i in range(len(vals) - 1)]
biggest, at = max(gaps)
runner_up = sorted(g for g, _ in gaps)[-2] if len(gaps) > 1 else 0

print(f"largest gap between adjacent answers : {biggest:,.1f}")
print(f"next largest                         : {runner_up:,.1f}")

if runner_up and biggest > 5 * runner_up:
    A, B = vals[:at+1], vals[at+1:]
    print(f"\nThe largest gap is {biggest/runner_up:.0f}x the next. These are two groups,")
    print("not one cloud. Almost certainly two different methods.\n")
    for name, grp in (("group 1", A), ("group 2", B)):
        m = np.mean(grp)
        print(f"  {name}:  n={len(grp):<3} {min(grp):,.1f}–{max(grp):,.1f}   "
              f"mean {m:,.1f}   error {m-truth:+,.1f} ({(m-truth)/truth*100:+.2f}%)")
    print(f"\n  separation between group means: {np.mean(B)-np.mean(A):,.1f}")
    print("\n  Read one response from each group. The gap has a cause, and the")
    print("  cause has a name. Put both in your findings.")
    tight = max(A, B, key=len)
    print(f"\n  Note also the spread *within* the larger group: "
          f"{max(tight)-min(tight):,.1f}.")
    print("  Even when it picks one method, it does not reproduce its own arithmetic.")
else:
    print("\nNo dominant gap — the answers look like one cloud rather than two")
    print("methods. Say so, and describe the shape you do see.")

exact = int((parsed - truth).abs().lt(0.005).sum())
close = int(((parsed - truth).abs() / truth < 0.005).sum())
print(f"\nexactly equal to the Python value : {exact} of {len(parsed)}")
print(f"within 0.5% of it                 : {close} of {len(parsed)}")
if close and not exact:
    print(f"\n{close} answers would survive a reviewer glancing at the number.")
    print("None of them is right.")

### Compare

You now have two ways of answering the same question. One of them gives the same answer every time.

In [ ]:
if len(parsed):
    err = (parsed - truth)
    comp = pd.DataFrame({
        "run": df.loc[df["parsed"], "run"].values,
        "answer": parsed.values,
        "error": err.values,
        "error_%": (err / truth * 100).values,
    })
    display(comp.style.format({"answer": "{:,.1f}", "error": "{:+,.1f}",
                               "error_%": "{:+.2f}%"}))

    within = (err.abs() / truth < 0.005).sum()
    print(f"within 0.5% of the Python result: {within} of {len(parsed)}")
    print(f"worst error: {err.abs().max():,.1f} HK$m "
          f"({err.abs().max()/truth*100:.1f}%)")
    print(f"\nIf you had taken run 1 and your colleague had taken the worst run,")
    print(f"you would differ by {abs(parsed.iloc[0] - parsed.loc[err.abs().idxmax()]):,.1f} HK$m "
          f"on the same company, on the same day, from the same prompt.")

---

## Read the failures

Aggregate statistics hide the interesting part. Open two or three of the responses that disagreed with the Python result and work out **why** each one differs. The errors are not random noise — they have names.

Look in particular for:

- the terminal value discounted for one period too many
- the final cash flow capitalised without being grown by `g` first
- discount factors rounded early, then compounded
- a convention applied that nobody asked for
- an input silently misread

Every one of these is a mistake a junior analyst makes, and every one of them is invisible in the final number.

In [ ]:
# Print the runs that disagreed most, so you can read them.
if len(parsed) and len(parsed) < len(df):
    print("=== responses with no parseable answer ===\n")
    for i, ok in enumerate(df["parsed"]):
        if not ok:
            print(f"--- run {i+1} ---\n{responses[i][:700]}\n")

if len(parsed) > 1:
    order = (parsed - truth).abs().sort_values(ascending=False).index[:2]
    for i in order:
        print(f"=== run {int(df.loc[i, 'run'])}: "
              f"{df.loc[i, 'answer']:,.1f} vs {truth:,.1f} ===")
        print(responses[i][:700], "\n")

---

## Calibration log

Fill this in **honestly**. Being wrong here costs you nothing; being wrong and not noticing is the thing this course is trying to train out of you. Systematic overconfidence is a finding I will mark as one.

In [ ]:
CALIBRATION = {
    # Before you ran anything, how many of ten runs did you expect to agree?
    "expected_identical_runs": None,      # 0-10
    # And how many actually did?
    "actual_distinct_answers": None,      # fill from the table above
    # Before checking, how much did you trust the model's answer? 1 = not at all,
    # 5 = would put my name on it.
    "trust_before": None,                 # 1-5
    "trust_after":  None,                 # 1-5
    # In one sentence: what surprised you?
    "surprise": "",
}

# What you found. Two or three sentences. Name the error types you identified.
FINDINGS = """
"""

# What you did NOT check, and why you are still willing to submit this.
RESIDUAL_RISK = """
"""

for k, v in CALIBRATION.items():
    if v in (None, ""):
        print(f"  still to fill in: {k}")

---

## Appendix and submission

The client has been recording every call. This generates the Reproducibility and Verification Appendix that the course outline requires — you do not write it by hand, but you do have to fill in the judgement parts above.

In [ ]:
from IPython.display import Markdown

md = fina4030a.appendix(
    student=f"{NAME} ({STUDENT_ID})",
    verification=FINDINGS.strip(),
    residual_risk=RESIDUAL_RISK.strip(),
    reproducibility=(
        f"Ten calls at temperature 0.0 produced "
        f"{int(df['answer'].nunique())} distinct answers and "
        f"{int((~df['parsed']).sum())} format failures. The Python "
        f"implementation produced one value on every run."
    ),
)
path = fina4030a.save_transcript("lab02_transcript.json")
display(Markdown(md))
print(f"\nTranscript written to {path}")

---

## What to submit

1. This notebook, **with outputs**, saved as `lab02_<your student ID>.ipynb`
2. `lab02_transcript.json`, written by the cell above

**If you ran against the cached responses** rather than a live provider, say so in one line in your findings. The cached responses are constructed for offline use, not recorded from a model — the lesson holds, but you did not measure anything, and claiming otherwise would be exactly the kind of misstatement the appendix exists to prevent.

---

### One thing to carry into Class 3

You used `temperature=0.0` — the setting that is supposed to make output deterministic. If your answers still varied, ask yourself where the remaining variation came from, because it is not a knob you can turn off. Class 3 is about the failure modes that do not announce themselves, and this is the first of them.
